# SCL vs no-SCL (F1 Comparison)

This notebook compares:
- Baseline run (no SCL)
- Best SCL run

It reports absolute and relative F1 improvement (delta).

In [34]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable

try:
    import torch
except Exception:
    torch = None

ROOT = Path.cwd()
print(f"Working directory: {ROOT}")

Working directory: /content


## 1) Configure Input Mode
Choose one mode:
- `manual`: enter baseline and SCL F1 directly.
- `auto`: scan files and try to extract F1 values automatically.

In [35]:
# --- USER CONFIG ---
# Toggle mode here:
MODE = "manual"  # use "manual" or "auto"

# Manual mode values (edit only these 2 lines when MODE="manual")
BASELINE_F1_MANUAL = 0.70
SCL_BEST_F1_MANUAL = 0.74

# Auto mode search settings (used only when MODE="auto")
SEARCH_ROOT = ROOT
MAX_FILES_TO_SCAN = 300

# Hints for identifying file groups in auto mode.
# Add your real naming patterns if needed.
BASELINE_HINTS = ["baseline", "no_scl", "no-scl", "without_scl", "vanilla", "base"]
SCL_HINTS = ["scl", "with_scl", "scl_run", "contrastive"]

# Quick switch guide:
# - For manual run: MODE = "manual"
# - For auto run  : MODE = "auto"

## 2) Helper Functions
Parse F1 values from JSON/TXT/LOG and optional checkpoint files.

In [36]:
F1_KEY_PATTERNS = [
    re.compile(r"^f1$", re.IGNORECASE),
    re.compile(r"val[_\- ]?f1", re.IGNORECASE),
    re.compile(r"macro[_\- ]?f1", re.IGNORECASE),
]

F1_IN_TEXT_PATTERN = re.compile(
    r"(?:val[_\- ]?f1|macro[_\- ]?f1|f1)\s*[:=]\s*([0-9]*\.?[0-9]+)",
    re.IGNORECASE,
)

def _looks_like_f1_key(key: str) -> bool:
    return any(p.search(str(key)) for p in F1_KEY_PATTERNS)

def _collect_f1_from_obj(obj: Any, out: list[float]) -> None:
    if isinstance(obj, dict):
        for k, v in obj.items():
            if _looks_like_f1_key(k) and isinstance(v, (int, float)):
                val = float(v)
                if 0.0 <= val <= 1.0:
                    out.append(val)
            _collect_f1_from_obj(v, out)
    elif isinstance(obj, list):
        for item in obj:
            _collect_f1_from_obj(item, out)

def _extract_f1_from_text(text: str) -> list[float]:
    vals = []
    for m in F1_IN_TEXT_PATTERN.finditer(text):
        val = float(m.group(1))
        if 0.0 <= val <= 1.0:
            vals.append(val)
    return vals

def extract_f1_candidates(path: Path) -> list[float]:
    suffix = path.suffix.lower()
    vals: list[float] = []

    if suffix in {".json", ".jsonl"}:
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
            if suffix == ".json":
                obj = json.loads(text)
                _collect_f1_from_obj(obj, vals)
            else:
                for line in text.splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        _collect_f1_from_obj(obj, vals)
                    except Exception:
                        vals.extend(_extract_f1_from_text(line))
        except Exception:
            pass

    elif suffix in {".txt", ".log", ".md", ".yaml", ".yml"}:
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
            vals.extend(_extract_f1_from_text(text))
        except Exception:
            pass

    elif suffix in {".pt", ".pth", ".ckpt"} and torch is not None:
        try:
            obj = torch.load(path, map_location="cpu")
            _collect_f1_from_obj(obj, vals)
        except Exception:
            pass

    return vals

def has_any_hint(path: Path, hints: Iterable[str]) -> bool:
    name = str(path).lower()
    return any(h.lower() in name for h in hints)

## 3) Resolve Baseline and Best SCL F1

In [37]:
def auto_resolve_f1(search_root: Path):
    patterns = ["*.json", "*.jsonl", "*.txt", "*.log", "*.md", "*.yaml", "*.yml", "*.pt", "*.pth", "*.ckpt"]
    files = []
    for p in patterns:
        files.extend(search_root.rglob(p))

    files = files[:MAX_FILES_TO_SCAN]

    baseline_vals = []
    scl_vals = []

    for fp in files:
        vals = extract_f1_candidates(fp)
        if not vals:
            continue

        if has_any_hint(fp, BASELINE_HINTS):
            baseline_vals.extend(vals)

        if has_any_hint(fp, SCL_HINTS):
            scl_vals.extend(vals)

    return baseline_vals, scl_vals

if MODE == "manual":
    baseline_f1 = float(BASELINE_F1_MANUAL)
    scl_best_f1 = float(SCL_BEST_F1_MANUAL)
    scan_info = "Manual mode: used user-provided values."

elif MODE == "auto":
    b_vals, s_vals = auto_resolve_f1(SEARCH_ROOT)
    if not b_vals:
        raise ValueError("Could not find baseline F1 automatically. Add hints or switch to manual mode.")
    if not s_vals:
        raise ValueError("Could not find SCL F1 automatically. Add hints or switch to manual mode.")

    baseline_f1 = max(b_vals)
    scl_best_f1 = max(s_vals)
    scan_info = f"Auto mode: baseline candidates={len(b_vals)}, SCL candidates={len(s_vals)}"

else:
    raise ValueError("MODE must be 'manual' or 'auto'.")

print(scan_info)
print(f"baseline_f1 = {baseline_f1:.6f}")
print(f"scl_best_f1 = {scl_best_f1:.6f}")

Manual mode: used user-provided values.
baseline_f1 = 0.700000
scl_best_f1 = 0.740000


## 4) Compute Improvement Delta

In [38]:
delta_abs = scl_best_f1 - baseline_f1
delta_rel_pct = (delta_abs / baseline_f1 * 100.0) if baseline_f1 != 0 else float("inf")

direction = "improvement" if delta_abs >= 0 else "drop"

print("=== SCL Comparison Report ===")
print(f"Baseline (no SCL) F1 : {baseline_f1:.4f}")
print(f"Best SCL run F1      : {scl_best_f1:.4f}")
print(f"Absolute delta        : {delta_abs:+.4f}")
print(f"Relative delta        : {delta_rel_pct:+.2f}%")
print(f"Result                : {direction}")

=== SCL Comparison Report ===
Baseline (no SCL) F1 : 0.7000
Best SCL run F1      : 0.7400
Absolute delta        : +0.0400
Relative delta        : +5.71%
Result                : improvement


## 5) Optional: One-line Summary

In [39]:
summary = (
    f"Baseline (no SCL) F1 = {baseline_f1:.4f}; "
    f"Best SCL F1 = {scl_best_f1:.4f}; "
    f"Delta = {delta_abs:+.4f} ({delta_rel_pct:+.2f}%)."
)
print(summary)

Baseline (no SCL) F1 = 0.7000; Best SCL F1 = 0.7400; Delta = +0.0400 (+5.71%).
